In [39]:
#imports
import os
import sys
import socket
import json
import numpy as np
import pandas as pd
from dateutil.parser import parse
from glob import glob

from utils import get_attributes_ukbbpuller, remove_outliers

In [40]:
# Paths (HPC)
outpth = "/project/ukbblatent/soumick/processed_clinical_data/F20208v2/v0.9.6_sixth_basket" #if not supplied, "processed" folder will be created in the same folder as the input file

pth_covs = "/group/glastonbury/GWAS/inputs/covariates/cov_F20208_Long_axis_heart_images_DICOM_H5v2.tsv"

pth_root_baselines = '/project/ukbblatent/clinicaldata'
pth_root_clinicaldata = '/project/ukbblatent/clinicaldata/v0.9.6_sixth_basket'

In [41]:
# params
thresh_notna = 0.3 #The column must have 30% non-NaN values to be considered

clinicaldata_prefixes = "aortameas;cardiacfunc;negcontrol" #coma-seperated valyes can be provided for multiple prefixes to be combined into one

In [42]:
# create complete paths
os.makedirs(outpth, exist_ok=True)

pth_baseline = f"{pth_root_baselines}/baseline.tsv"
pth_dob = f"{pth_root_baselines}/dateofbirth.tsv"
pth_body_measure = f"{pth_root_baselines}/body_measure.tsv"

In [43]:
# read and process covariates
covariates = pd.read_table(pth_covs, index_col=0)
covariates.drop(['IID'], axis=1, inplace=True)

In [44]:
# UKBBcols
cols = pd.read_table("../../UKBB/fields_meta.tsv", index_col="field_id")
cols.title = cols.title.str.replace(" ", "_")
cols.index = "f."+cols.index.astype(str)+"."
cols.title = cols.apply(lambda row: "CAT_" + row.title if row.value_type == 21 else row.title, axis=1)
cols_dict = cols.to_dict()["title"]

# value_type
# 31     Continuous
# 21     Categorical (single)
# 51     Date
# 11     Integer
# 41     Text
# 22     Categorical (multiple)
# 61     Time
# 101    Compound

In [45]:
# process baseline

baseline = pd.read_table(pth_baseline).set_index('f.eid')
dob = pd.read_table(pth_dob).set_index('f.eid')
baseline = dob.join(baseline)

cols_nodrop = ['f.40000.', 'f.40001.'] # need to ignore death columns while processing baseline based on threshold
baseline = get_attributes_ukbbpuller(file_path="", df=baseline, indices=covariates.index, thresh_notna=thresh_notna, cols_nodrop=cols_nodrop, col_names=cols_dict, custom_dict={}, default_order=[2,3,1,0,4,5,6])

baseline.rename(columns={'CAT_Smoking_status': 'CAT_Smoking', 
                         'CAT_Ethnic_background': 'CAT_Ethnicity', 
                         'Body_mass_index_(BMI)': 'BMI',
                         'CAT_Underlying_(primary)_cause_of_death:_ICD10': 'CAT_Death_Cause'}, inplace=True)

baseline['CAT_Smoking'] = baseline['CAT_Smoking'].replace(-3, np.NaN) # remove -3 values (unknown answer)
baseline = baseline[baseline['CAT_Smoking'].notna()]

baseline['CAT_BMI'] = pd.cut(baseline['BMI'], bins=[0,18.5,24.9,29.9,60], labels=['Underweight', 'Healthy', 'Overweight', 'Obese'])

# add categories for Ethnicity
# following data coding 1001 (https://biobank.ctsu.ox.ac.uk/crystal/coding.cgi?id=1001)
ethnic_dict = {'1':'White', '2':'Mixed', '3':'S.Asian', '4':'Black', '5':'E.Asian', '6':'Other', '-':'Other', 'n':'Other'}
baseline['CAT_Ethnicity'] = baseline['CAT_Ethnicity'].apply(lambda x: ethnic_dict.get(str(x)[0]))

# Merge baseline with covariates
baseline = baseline.join(covariates, lsuffix='', rsuffix='_2del')
baseline.drop([col for col in baseline.columns if "_2del" in col], axis=1, inplace=True)

baseline.drop(["Gender"], inplace=True, axis=1)
baseline.rename({'CAT_Sex': 'BinCAT_Sex', 'MRI_Visit': 'BinCAT_MRI_Visit', 'MRI_Centre': 'CAT_MRI_Centre'}, axis=1, inplace=True)

baseline_alive = baseline[baseline['Date_of_death'].isna()].drop(['Date_of_death', 'CAT_Death_Cause'], axis=1)
baseline_alive.dropna().to_csv(f"{outpth}/baseline_alive.tsv", sep="\t")

baseline_dead = baseline[baseline['Date_of_death'].notna()]
baseline_dead.dropna().to_csv(f"{outpth}/baseline_dead.tsv", sep="\t")

baseline = baseline.drop(['Date_of_death', 'CAT_Death_Cause'], axis=1) # just going to ignore the death columns
baseline.dropna().to_csv(f"{outpth}/baseline.tsv", sep="\t")

/scratch/soumick.chatterjee/conda_envs/BeegFSTorchHTBeta2/lib/python3.10/site-packages/pandas/core/indexes/base.py:6999: FutureWarning: In a future version, the Index constructor will not infer numeric dtypes when passed object-dtype sequences (matching Series behavior)
  return Index(sequences[0], name=names)


In [46]:
body_measure_cols2drop = ['f.3077.', 'f.21.', 'f.12143.', 'f.12144.', 'f.21001.']
body_measure = get_attributes_ukbbpuller(file_path=pth_body_measure, indices=covariates.index, thresh_notna=thresh_notna, cols2drop=body_measure_cols2drop, col_names=cols_dict, custom_dict={}, default_order=[2,3,1,0,4,5,6])

baseline_bodymeas = baseline.join(body_measure, lsuffix='', rsuffix='_2del')
baseline_bodymeas.drop([col for col in baseline_bodymeas.columns if "_2del" in col], axis=1, inplace=True)

baseline_bodymeas.dropna().to_csv(f"{outpth}/baseline_bodymeas.tsv", sep="\t")

In [47]:
pths_tsvs = glob(f"{pth_root_clinicaldata}/*.tsv")
clinicaldata_prefixes = clinicaldata_prefixes.split(';') if isinstance(clinicaldata_prefixes, str) else clinicaldata_prefixes
for prefixes in clinicaldata_prefixes:
    prefixes = prefixes.split(',')
    filtered_paths = [path for path in pths_tsvs if any(os.path.basename(path).startswith(prefix) for prefix in prefixes)]
    print(f"Found {len(filtered_paths)} files with prefixes {prefixes}")
    df = pd.concat([pd.read_table(path, index_col='f.eid') for path in filtered_paths], axis=1)
    
    df = get_attributes_ukbbpuller(file_path="", df=df, indices=covariates.index, thresh_notna=thresh_notna, col_names=cols_dict, custom_dict={}, default_order=[2,3,1,0,4,5,6])
    df = remove_outliers(df, contamination=0.01, support_fraction=0.63)
    
    df.to_csv(f"{outpth}/clinicaldata_{prefixes[0]}.tsv", sep="\t")

Found 1 files with prefixes ['aortameas']
Found 8 files with prefixes ['cardiacfunc']
Found 1 files with prefixes ['negcontrol']


In [48]:
# Edit the files using knowledge about the data [Maintain Caution]

# negcontrol
clinicaldata_negcontrol = pd.read_table(f"{outpth}/clinicaldata_negcontrol.tsv", index_col='f.eid')
clinicaldata_negcontrol['CAT_Risk_taking'] = clinicaldata_negcontrol['CAT_Risk_taking'].replace(-3, np.NaN) # remove -3 values (Prefer not to answer)
clinicaldata_negcontrol['CAT_Risk_taking'] = clinicaldata_negcontrol['CAT_Risk_taking'].replace(-1, np.NaN) # remove -1 values (Do not know)
clinicaldata_negcontrol['CAT_Hair_colour_(natural,_before_greying)'] = clinicaldata_negcontrol['CAT_Hair_colour_(natural,_before_greying)'].replace(-3, np.NaN) # remove -3 values (Prefer not to answer)
clinicaldata_negcontrol['CAT_Hair_colour_(natural,_before_greying)'] = clinicaldata_negcontrol['CAT_Hair_colour_(natural,_before_greying)'].replace(-1, np.NaN) # remove -1 values (Do not know)
clinicaldata_negcontrol['CAT_Hair_colour_(natural,_before_greying)'] = clinicaldata_negcontrol['CAT_Hair_colour_(natural,_before_greying)'].replace(6, np.NaN) # remove 6 values (Other)
clinicaldata_negcontrol.dropna().rename({'CAT_Hair_colour_(natural,_before_greying)': 'CAT_Nat_Hair_colour', 
                                         'CAT_Risk_taking': 'BinCAT_Risk_taking'}, axis=1).to_csv(f"{outpth}/clinicaldata_negcontrol.tsv", sep="\t")